[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-3-nlp-to-transformers/01-text-as-data/code/text_as_data.ipynb)

# Class 3.1: Text as data

The slides carry the ideas. Here you turn text into tokens and ids by hand, build a tiny byte-pair encoder, and see that tokenization is a choice, not a fixed fact.

What we will cover:
- a whitespace tokenizer, and token-to-id mapping
- one-hot (the naive vector) and an nn.Embedding lookup
- byte-pair encoding on a small corpus: build a vocabulary and tokenize new words
- two tokenizers splitting the same text differently
- why a bag of tokens loses word order


## A whitespace tokenizer

The simplest tokenizer: lowercase, then split on spaces. Real tokenizers are subtler, but this shows the shape.

In [1]:
def whitespace_tokenize(text):
    return text.lower().split()

sentence = "The cat sat on the mat"
tokens = whitespace_tokenize(sentence)
print("tokens:", tokens)
print("token count:", len(tokens))

tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']
token count: 6


## Token to id, and back

A vocabulary maps each token to an integer id. The id is just a name; the meaning-carrying vector comes later (the embedding).

In [2]:
vocab = {tok: i for i, tok in enumerate(sorted(set(tokens)))}
ids = [vocab[t] for t in tokens]
inv = {i: t for t, i in vocab.items()}
print("vocab:", vocab)
print("ids:", ids)
print("decoded:", [inv[i] for i in ids])

vocab: {'cat': 0, 'mat': 1, 'on': 2, 'sat': 3, 'the': 4}
ids: [4, 0, 3, 2, 4, 1]
decoded: ['the', 'cat', 'sat', 'on', 'the', 'mat']


## One-hot: the naive vector

The simplest way to turn an id into a vector is one-hot: a vector as long as the whole vocabulary, all zeros except a single 1. It works, but it is huge and sparse, and every token sits the same distance from every other.

In [3]:
import numpy as np

onehot = np.zeros((len(tokens), len(vocab)), dtype=int)
for row, t in enumerate(tokens):
    onehot[row, vocab[t]] = 1

print("one-hot matrix shape:", onehot.shape)   # (n_tokens, vocab_size)
print("row for", repr(tokens[0]), "->", onehot[0])
print("nonzeros per row:", onehot.sum(axis=1).tolist())

one-hot matrix shape: (6, 5)
row for 'the' -> [0 0 0 0 1]
nonzeros per row: [1, 1, 1, 1, 1, 1]


## An embedding is a lookup table

Replace the giant sparse one-hot with a short, dense, learned vector. An embedding layer is one matrix, one row per token; the id just selects a row. In PyTorch that is `nn.Embedding`, and its numbers are learned like any weight.

In [5]:
import torch
import torch.nn as nn

torch.manual_seed(0)
dim = 4                                  # a tiny embedding size
emb = nn.Embedding(len(vocab), dim)      # table shape: (vocab_size, dim)

ids_t = torch.tensor(ids)
vectors = emb(ids_t)                     # id -> row, a dense vector
print("embedding table shape:", tuple(emb.weight.shape))
print("id", ids[0], "->", vectors[0].detach().round(decimals=2).tolist())
print("dense length", dim, "vs one-hot length", len(vocab))

embedding table shape: (5, 4)
id 4 -> [0.9300000071525574, 1.2599999904632568, 2.0, 0.05000000074505806]
dense length 4 vs one-hot length 5


## Byte-pair encoding: build a vocabulary from a corpus

Start from a few sentences. Split into words, write each word as characters plus an end-of-word marker `</w>`, then repeatedly merge the most frequent adjacent pair.

In [6]:
from collections import Counter

corpus = [
    "the cat sat on the mat",
    "the cat saw the dog",
    "the dog sat on the log",
    "a cat and a dog",
]

word_freqs = Counter(w for s in corpus for w in s.split())
vocab = {" ".join(list(w)) + " </w>": f for w, f in word_freqs.items()}
base_vocab = {sym for word in vocab for sym in word.split()}   # the starting alphabet

print("word counts:", dict(word_freqs))
print("base characters:", len(base_vocab))

word counts: {'the': 6, 'cat': 3, 'sat': 2, 'on': 2, 'mat': 1, 'saw': 1, 'dog': 3, 'log': 1, 'a': 2, 'and': 1}
base characters: 14


In [7]:
base_vocab

{'</w>', 'a', 'c', 'd', 'e', 'g', 'h', 'l', 'm', 'n', 'o', 's', 't', 'w'}

## Merge to a target vocabulary size

Each step merges the single most frequent adjacent pair (no cutoff). The vocabulary is the base characters plus one new token per merge, so it grows by one each step. Stop when it reaches the target size.

In [10]:
def get_stats(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        syms = word.split()
        for a, b in zip(syms, syms[1:]):
            pairs[(a, b)] += freq
    return pairs

def merge_pair(pair, vocab):
    bigram, joined = " ".join(pair), "".join(pair)
    return {w.replace(bigram, joined): f for w, f in vocab.items()}

target_size = 30
merges = []
def vocab_size():
    return len(base_vocab | {"".join(m) for m in merges})

while vocab_size() < target_size:
    stats = get_stats(vocab)
    if not stats:
        break
    best = max(stats, key=stats.get)                 # most frequent pair
    vocab = merge_pair(best, vocab)
    merges.append(best)
    print(f"merge {len(merges):2d}: {best[0]} + {best[1]} -> {''.join(best)!r}  (seen {stats[best]}x)")

print("final vocabulary size:", vocab_size(), "=", len(base_vocab), "chars +", len(merges), "merges")

merge  1: t + h -> 'th'  (seen 6x)
merge  2: th + e -> 'the'  (seen 6x)
merge  3: the + </w> -> 'the</w>'  (seen 6x)
merge  4: a + t -> 'at'  (seen 6x)
merge  5: at + </w> -> 'at</w>'  (seen 6x)
merge  6: o + g -> 'og'  (seen 4x)
merge  7: og + </w> -> 'og</w>'  (seen 4x)
merge  8: c + at</w> -> 'cat</w>'  (seen 3x)
merge  9: d + og</w> -> 'dog</w>'  (seen 3x)
merge 10: s + at</w> -> 'sat</w>'  (seen 2x)
merge 11: o + n -> 'on'  (seen 2x)
merge 12: on + </w> -> 'on</w>'  (seen 2x)
merge 13: a + </w> -> 'a</w>'  (seen 2x)
merge 14: m + at</w> -> 'mat</w>'  (seen 1x)
merge 15: s + a -> 'sa'  (seen 1x)
merge 16: sa + w -> 'saw'  (seen 1x)
final vocabulary size: 30 = 14 chars + 16 merges


## Tokenize new words with the learned merges

Common whole words have become single tokens. A word the corpus never saw still tokenizes, into known subword pieces.

In [11]:
final_vocab = sorted(base_vocab | {"".join(m) for m in merges})
whole_words = [t for t in final_vocab if t.endswith("</w>") and len(t) > 4]
print("whole-word tokens:", whole_words)

def encode(word, merges):
    syms = list(word) + ["</w>"]
    for a, b in merges:
        out, i = [], 0
        while i < len(syms):
            if i + 1 < len(syms) and syms[i] == a and syms[i+1] == b:
                out.append(a + b); i += 2
            else:
                out.append(syms[i]); i += 1
        syms = out
    return syms

print("encode 'cats':", encode("cats", merges))   # unseen word -> subwords
print("encode 'dogs':", encode("dogs", merges))

whole-word tokens: ['a</w>', 'at</w>', 'cat</w>', 'dog</w>', 'mat</w>', 'og</w>', 'on</w>', 'sat</w>', 'the</w>']
encode 'cats': ['c', 'at', 's', '</w>']
encode 'dogs': ['d', 'og', 's', '</w>']


## Two tokenizers, different splits

Tokenization is a design choice. Here a whitespace tokenizer and a suffix-splitting tokenizer disagree on the same sentence.

In [12]:
SUFFIXES = ("ing", "ly", "ed", "s")

def suffix_tokenize(text):          # mark a continued piece with ## (WordPiece style)
    out = []
    for w in text.lower().split():
        for suf in SUFFIXES:
            if w.endswith(suf) and len(w) > len(suf) + 1:
                out += [w[:-len(suf)], "##" + suf]
                break
        else:
            out.append(w)
    return out

s = "the cats are running quickly"
print("whitespace:", whitespace_tokenize(s), "->", len(whitespace_tokenize(s)), "tokens")
print("suffix    :", suffix_tokenize(s), "->", len(suffix_tokenize(s)), "tokens")

whitespace: ['the', 'cats', 'are', 'running', 'quickly'] -> 5 tokens
suffix    : ['the', 'cat', '##s', 'are', 'runn', '##ing', 'quick', '##ly'] -> 8 tokens


## Why a bag of tokens loses order

If we treat a sentence as an unordered bag of tokens, two very different sentences can look identical.

In [13]:
a = "dog bites man"
b = "man bites dog"
print("same bag of tokens:", sorted(a.split()) == sorted(b.split()))
print("but the sentences mean different things: order carries meaning")

same bag of tokens: True
but the sentences mean different things: order carries meaning


## Your turn

**Micro-assignment.** Six problems on tokenization; see `../micro-assignment/README.md`.

**Next, class 3.2 (Attention):** the mechanism that lets every token look at every other token, so word order and context are captured.